# Ingestão Bronze


In [0]:
import re
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp, current_timestamp


# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")



In [0]:
# Mapeamento explícito arquivo -> nome de tabela.
expected_files = {
    "base_atendentes.csv":      ("base_atendentes",      "UTF-8"),
    "base_motivos.csv":         ("base_motivos",         "UTF-8"),
    "canais.csv":               ("canais",               "UTF-8"),
    "chamados.csv":             ("chamados",             "UTF-8"),
    "chamados_hora.csv":        ("chamados_hora",        "ISO-8859-1"),
    "clientes.csv":             ("clientes",             "ISO-8859-1"),
    "custos.csv":               ("custos",               "UTF-8"),
    "pesquisa_satisfacao.csv":  ("pesquisa_satisfacao",  "UTF-8"),
}

for file, (name, encoding) in expected_files.items():
    table = f"{BRONZE_SCHEMA}.tb_{name}"
    file_path = f"{LANDING_PATH}/{file}"

    # Leitura: CSV bruto, sem inferência de schema.
    # Todas as colunas (não de metadados) entram como string.
    # Coluna referente a hora da ingestão, o _ antes do nome explicita ser uma coluna de metadados.
    df = spark.read.csv(file_path, header=True, encoding=encoding).withColumn(
        "_ingerido_em", current_timestamp()
    )

    # Escrita: Delta, modo overwrite (batelada única, sem ingestão incremental)
    #    e overwriteSchema para acomodar variações de tipo entre reexecuções.
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )

    # Validação: tabela vazia é erro
    count = spark.table(table).count()
    if count == 0:
        print(f"{FAIL} {table} foi gravada vazia")
        raise Exception(
            f"Tabela {table} vazia. Verifique o arquivo {file} na landing zone."
        )
    else:
        print(f"{OK} {table}: {count} linhas")

    # Inspeção visual: amostra das 5 primeiras linhas para conferência rápida.
    display(spark.table(table).limit(5))